# AI-Powered Task Management System - End-to-End ML Pipeline

This notebook demonstrates the full ML pipeline for predicting task priority based on description.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

## Phase 1: Data Processing & EDA

In [ ]:
df = pd.read_csv('../data/task_dataset.csv')
print('Data shape:', df.shape)
print('\nMissing Values: \n', df.isnull().sum())
df.head()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='priority', order=['High', 'Medium', 'Low'])
plt.title('Priority Distribution')
plt.show()

## Phase 2: NLP Preprocessing

In [ ]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['task_description'].apply(preprocess_text)
df[['task_description', 'clean_text']].head()

## Phase 3 & 4: Feature Engineering & Model Building

In [ ]:
X = df['clean_text']
y = df['priority']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=5000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

models = {
    'Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

## Phase 5: Model Evaluation

In [ ]:
best_model_name = ""
best_acc = 0
best_model = None

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_model = model
        best_model_name = name

print(f"\nBest Model: {best_model_name}")

y_pred_best = best_model.predict(X_test_vec)
print(classification_report(y_test, y_pred_best))

cm = confusion_matrix(y_test, y_pred_best, labels=['High', 'Medium', 'Low'])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['High', 'Medium', 'Low'], yticklabels=['High', 'Medium', 'Low'], cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Phase 6: Model Saving

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
print(f"Best model ({best_model_name}) and vectorizer saved successfully!")